In [1]:
import stim

In [2]:
import torch
import torch.nn.functional as F


In [3]:
from torch_geometric.nn import MessagePassing, global_mean_pool
from torch_geometric.data import Data, Batch
from torch_geometric.utils import add_self_loops, degree
from torch_geometric.utils import softmax  # For attention normalization



In [4]:
import matplotlib.pyplot as plt


In [5]:
import numpy as np
from tqdm.auto import trange
import networkx as nx
import math

c:\Users\Win10\anaconda3\envs\gtx1080-eng\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [7]:
# ============================================================================
# PART 1: SURFACE CODE CIRCUIT
# ============================================================================

def surface_code_circuit(p, d):
    """Generate surface code circuit with given error rate and distance"""
    return stim.Circuit.generated(
        "surface_code:rotated_memory_z",
        rounds=d,
        distance=d,
        after_clifford_depolarization=p,
        after_reset_flip_probability=p,
        before_measure_flip_probability=p,
        before_round_data_depolarization=p
    )

In [ ]:
class SyndromeGraph:
    """
    Pre-built graph structure for a surface code.
    NOW TRACKS EDGE TYPES: 0=spatial, 1=temporal
    AND INCLUDES RICH NODE FEATURES: stabilizer_type, position, time
    """

    def __init__(self, circuit, device):
        self.device = device
        self.num_nodes = circuit.num_detectors

        # Build the fixed graph skeleton ONCE (now returns edge types too)
        self.edge_index, self.edge_types = self._build_edges(circuit)
        self.node_features = self._build_node_features(circuit)  # Static features

        self.edge_index = self.edge_index.to(device)
        self.edge_types = self.edge_types.to(device)
        self.node_features = self.node_features.to(device)

        self.num_edges = self.edge_index.shape[1]
        self.num_node_features = self.node_features.shape[1] + 1  # +1 for syndrome

        print(f"✓ Graph skeleton built: {self.num_nodes} detectors, {self.num_edges} connections")
        print(f"  Spatial edges: {(self.edge_types == 0).sum().item()}")
        print(f"  Temporal edges: {(self.edge_types == 1).sum().item()}")
        print(f"  Node features: {self.num_node_features} (syndrome + {self.node_features.shape[1]} static)")
        print(f"  Average neighbors per detector: {self.num_edges / self.num_nodes:.1f}")

    def _build_node_features(self, circuit):
        """
        Build static node features from detector coordinates.

        Features per node:
        - stabilizer_type: 0=X, 1=Z (based on checkerboard pattern)
        - dist_north: normalized distance from north boundary
        - dist_west: normalized distance from west boundary
        - time_round: normalized time round
        """
        detector_coords = circuit.get_detector_coordinates()

        # Get coordinate ranges for normalization
        all_coords = list(detector_coords.values())
        xs = [c[0] for c in all_coords]
        ys = [c[1] for c in all_coords]
        ts = [c[2] for c in all_coords]

        x_min, x_max = min(xs), max(xs)
        y_min, y_max = min(ys), max(ys)
        t_max = max(ts)

        features = []
        for det_id in range(self.num_nodes):
            coord = detector_coords.get(det_id, [0, 0, 0])
            x, y, t = coord[0], coord[1], coord[2]

            # Stabilizer type: In rotated surface code, X and Z alternate
            # Based on checkerboard pattern of (x + y) / 2
            stabilizer_type = ((x + y) / 2) % 2  # 0 or 1

            # Normalized distances from boundaries (0 to 1)
            dist_north = (y - y_min) / max(1, y_max - y_min)
            dist_west = (x - x_min) / max(1, x_max - x_min)

            # Normalized time round (0 to 1)
            time_norm = t / max(1, t_max)

            features.append([stabilizer_type, dist_north, dist_west, time_norm])

        return torch.tensor(features, dtype=torch.float)

    def _build_edges(self, circuit):
        """
        Build edge connections AND edge types.
        Returns: (edge_index, edge_types)
        """
        detector_coords = circuit.get_detector_coordinates()
        edges = []
        edge_types = []  # 0 = spatial, 1 = temporal

        # Build lookup: (x, y, t) -> detector_id for temporal connections
        coord_to_id = {}
        for det_id, coord in detector_coords.items():
            key = (coord[0], coord[1], coord[2])
            coord_to_id[key] = det_id

        for i in range(self.num_nodes):
            coord_i = detector_coords.get(i, [0, 0, 0])
            xi, yi, ti = coord_i[0], coord_i[1], coord_i[2]

            for j in range(i + 1, self.num_nodes):
                coord_j = detector_coords.get(j, [0, 0, 0])
                xj, yj, tj = coord_j[0], coord_j[1], coord_j[2]

                # Spatial connection: same time round, adjacent in space
                if ti == tj:
                    dx = abs(xi - xj)
                    dy = abs(yi - yj)

                    if (dx == 2 and dy == 0) or (dx == 0 and dy == 2):
                        edges.append([i, j])
                        edges.append([j, i])
                        edge_types.extend([0, 0])  # Both directions are spatial

            # Temporal connection: same spatial position, next time round
            next_key = (xi, yi, ti + 1)
            if next_key in coord_to_id:
                j = coord_to_id[next_key]
                edges.append([i, j])
                edge_types.append(1)  # Temporal

        if len(edges) == 0:
            print("  Warning: No edges found, using fallback chain")
            for i in range(self.num_nodes - 1):
                edges.append([i, i + 1])
                edges.append([i + 1, i])
                edge_types.extend([0, 0])

        edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
        edge_types = torch.tensor(edge_types, dtype=torch.long)
        return edge_index, edge_types

    def create_batch(self, syndrome_values):
        """
        Create batched graph data from syndrome measurements.
        NOW INCLUDES RICH NODE FEATURES: [syndrome, stabilizer_type, dist_north, dist_west, time]
        """
        batch_size = syndrome_values.shape[0]

        # Syndrome values: [batch, nodes] -> [batch*nodes, 1]
        syndrome_feat = syndrome_values[:, :self.num_nodes].reshape(-1, 1).float()

        # Static features: repeat for each sample in batch [batch*nodes, 4]
        static_feat = self.node_features.repeat(batch_size, 1)

        # Concatenate all features: [batch*nodes, 5]
        x = torch.cat([syndrome_feat, static_feat], dim=1)

        # Replicate edges and edge types for each graph in batch
        edge_indices = []
        edge_types_list = []
        for i in range(batch_size):
            offset = i * self.num_nodes
            edge_indices.append(self.edge_index + offset)
            edge_types_list.append(self.edge_types)

        edge_index = torch.cat(edge_indices, dim=1)
        edge_type = torch.cat(edge_types_list, dim=0)

        # Batch assignment
        batch = torch.arange(batch_size, device=self.device).repeat_interleave(self.num_nodes)

        return Data(x=x, edge_index=edge_index, edge_type=edge_type, batch=batch)


In [9]:
class GATConvWithEdgeType(MessagePassing):
    """
    Graph Attention Layer with edge type embeddings.

    Attention score for edge i->j with type t:
        α_ij = softmax(LeakyReLU(a^T [W*h_i || W*h_j || edge_emb_t]))
    """

    def __init__(self, in_channels, out_channels, heads=4, concat=True,
                 dropout=0.0, num_edge_types=2):
        super().__init__(aggr='add', node_dim=0)

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.heads = heads
        self.concat = concat
        self.dropout = dropout
        self.num_edge_types = num_edge_types

        # Linear transformation for node features
        self.lin = torch.nn.Linear(in_channels, heads * out_channels, bias=False)

        # Edge type embeddings
        self.edge_type_emb = torch.nn.Embedding(num_edge_types, heads * out_channels)

        # Attention mechanism: a^T [source || target || edge_type]
        self.att = torch.nn.Parameter(torch.Tensor(1, heads, 3 * out_channels))

        # Bias
        if concat:
            self.bias = torch.nn.Parameter(torch.Tensor(heads * out_channels))
        else:
            self.bias = torch.nn.Parameter(torch.Tensor(out_channels))

        self.reset_parameters()

    def reset_parameters(self):
        torch.nn.init.xavier_uniform_(self.lin.weight)
        torch.nn.init.xavier_uniform_(self.att)
        torch.nn.init.xavier_uniform_(self.edge_type_emb.weight)
        torch.nn.init.zeros_(self.bias)

    def forward(self, x, edge_index, edge_type):
        """
        Args:
            x: Node features [num_nodes, in_channels]
            edge_index: Edge indices [2, num_edges]
            edge_type: Edge types [num_edges] (0=spatial, 1=temporal)
        """
        # Transform node features
        x = self.lin(x).view(-1, self.heads, self.out_channels)

        # Get edge type embeddings
        edge_emb = self.edge_type_emb(edge_type).view(-1, self.heads, self.out_channels)

        # Start propagating messages
        out = self.propagate(edge_index, x=x, edge_emb=edge_emb)

        # Concatenate or average heads
        if self.concat:
            out = out.view(-1, self.heads * self.out_channels)
        else:
            out = out.mean(dim=1)

        return out + self.bias

    def message(self, x_i, x_j, edge_emb, index, ptr, size_i):
        """
        Compute attention coefficients and messages.

        x_i: Target node features [num_edges, heads, out_channels]
        x_j: Source node features [num_edges, heads, out_channels]
        edge_emb: Edge type embeddings [num_edges, heads, out_channels]
        """
        # Concatenate source, target, and edge type: [num_edges, heads, 3*out_channels]
        alpha_input = torch.cat([x_i, x_j, edge_emb], dim=-1)

        # Compute attention coefficients: [num_edges, heads]
        alpha = (alpha_input * self.att).sum(dim=-1)
        alpha = F.leaky_relu(alpha, negative_slope=0.2)
        alpha = softmax(alpha, index, ptr, size_i)
        alpha = F.dropout(alpha, p=self.dropout, training=self.training)

        # Apply attention to source features
        return x_j * alpha.unsqueeze(-1)


In [ ]:

class SyndromeDecoder(torch.nn.Module):
    """
    GNN with attention that takes syndrome graphs and predicts logical error.
    Now accepts 5 input features: [syndrome, stabilizer_type, dist_north, dist_west, time]
    """

    def __init__(self, hidden_dim=128, num_layers=4, num_heads=4, num_input_features=5):
        super().__init__()

        # Attention layers
        self.layers = torch.nn.ModuleList()
        self.norms = torch.nn.ModuleList()

        # First layer: num_input_features -> hidden
        self.layers.append(GATConvWithEdgeType(
            num_input_features, hidden_dim // num_heads,
            heads=num_heads,
            concat=True,
            num_edge_types=2
        ))
        self.norms.append(torch.nn.BatchNorm1d(hidden_dim))

        # Hidden layers
        for _ in range(num_layers - 1):
            self.layers.append(GATConvWithEdgeType(
                hidden_dim, hidden_dim // num_heads,
                heads=num_heads,
                concat=True,
                num_edge_types=2
            ))
            self.norms.append(torch.nn.BatchNorm1d(hidden_dim))

        # Output: aggregate graph -> predict error
        self.output = torch.nn.Sequential(
            torch.nn.Linear(hidden_dim, 64),
            torch.nn.SiLU(),
            torch.nn.Linear(64, 1)
        )

    def forward(self, data):
        x, edge_index, edge_type, batch = data.x, data.edge_index, data.edge_type, data.batch

        # Attention-based message passing
        for layer, norm in zip(self.layers, self.norms):
            x = layer(x, edge_index, edge_type)
            x = norm(x)
            x = F.silu(x)

        # Pool: aggregate all nodes in each graph
        graph_features = global_mean_pool(x, batch)

        # Predict logical error
        return self.output(graph_features)

In [11]:
# ============================================================================
# PART 5: GNN TRAINER - Clean interface with progress tracking
# ============================================================================

class GNNTrainer:
    """Clean interface for training GNN decoder with clear progress tracking"""

    def __init__(self, p, d, device, hidden_dim=128, num_layers=4):
        self.p = p
        self.d = d
        self.device = device

        print(f"\n{'='*60}")
        print(f"Initializing GNN Trainer for d={d}, p={p}")
        print(f"{'='*60}")

        # Build circuit (ONCE)
        print("\n[1/3] Building quantum circuit...")
        self.circuit = surface_code_circuit(p, d)

        # Build graph structure (ONCE)
        print("\n[2/3] Building graph skeleton...")
        self.graph = SyndromeGraph(self.circuit, device)

        # Build model
        print("\n[3/3] Building neural network...")
        self.model = SyndromeDecoder(hidden_dim=hidden_dim, num_layers=num_layers).to(device)
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=3e-4)
        # Note: loss_fn will be set dynamically in train_epoch to handle class imbalance

        # Count parameters
        num_params = sum(p.numel() for p in self.model.parameters())
        print(f"✓ Model built: {num_params:,} parameters")
        print(f"\n{'='*60}")
        print("Ready to train!")
        print(f"{'='*60}\n")

    def sample_data(self, num_samples):
        """Generate syndrome samples"""
        sampler = self.circuit.compile_detector_sampler()
        detections, flips = sampler.sample(num_samples, separate_observables=True)

        # Convert: 0/1 -> -1/+1
        # Use .copy() to ensure contiguous array, then torch.from_numpy() to avoid dtype inference issues
        syndrome_np = (detections.astype(np.int64) * 2 - 1).copy()
        labels_np = flips.astype(np.int64).flatten().copy()

        syndromes = torch.from_numpy(syndrome_np).to(self.device)
        labels = torch.from_numpy(labels_np).to(self.device)

        return syndromes, labels

    def train_epoch(self, syndromes, labels, batch_size=256):
        """One training epoch with progress bar"""
        self.model.train()
        num_samples = syndromes.shape[0]
        num_batches = math.ceil(num_samples / batch_size)

        # Compute class weights for imbalanced data
        # At low error rates, positive class (errors) is rare
        num_pos = labels.sum().item()
        num_neg = num_samples - num_pos
        if num_pos > 0 and num_neg > 0:
            pos_weight = torch.tensor([num_neg / num_pos], device=self.device)
        else:
            pos_weight = torch.tensor([1.0], device=self.device)
        loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

        total_loss = 0
        correct = 0
        samples_seen = 0

        with trange(num_batches, desc="Training", leave=False) as pbar:
            for batch_idx in pbar:
                start = batch_idx * batch_size
                end = min(start + batch_size, num_samples)  # Handle last batch

                batch_syn = syndromes[start:end]
                batch_lab = labels[start:end]
                batch_len = end - start  # Actual batch size

                # Create graph batch (structure pre-built, just plug in values)
                batch_data = self.graph.create_batch(batch_syn)

                # Forward (get logits, not probabilities)
                logits = self.model(batch_data)
                loss = loss_fn(logits, batch_lab.unsqueeze(1).float())

                # Backward
                self.optimizer.zero_grad()
                loss.backward()
                self.optimizer.step()

                # Track metrics (convert logits to predictions)
                total_loss += loss.item()
                pred = torch.sigmoid(logits)
                batch_correct = ((pred > 0.5).flatten() == batch_lab).sum().item()
                correct += batch_correct
                samples_seen += batch_len

                # Update progress bar with correct running accuracy
                running_acc = correct / samples_seen
                pbar.set_postfix({
                    'loss': f'{loss.item():.4f}',
                    'acc': f'{running_acc:.4f}'
                })

        avg_loss = total_loss / num_batches
        accuracy = correct / num_samples  # Use actual sample count
        return avg_loss, accuracy

    def evaluate(self, num_samples=10000, batch_size=500):
        """Evaluate on fresh data"""
        self.model.eval()
        syndromes, labels = self.sample_data(num_samples)

        all_preds = []
        with torch.no_grad():
            for i in range(0, num_samples, batch_size):
                batch_syn = syndromes[i:i+batch_size]
                batch_data = self.graph.create_batch(batch_syn)
                logits = self.model(batch_data)
                pred = torch.sigmoid(logits)
                all_preds.append((pred > 0.5).flatten())

        all_preds = torch.cat(all_preds)
        accuracy = (all_preds == labels).float().mean().item()
        return accuracy

    def get_ler(self, num_shots=100000, batch_size=1000):
        """Compute logical error rate"""
        self.model.eval()
        syndromes, labels = self.sample_data(num_shots)

        all_preds = []
        with torch.no_grad():
            for i in range(0, num_shots, batch_size):
                batch_syn = syndromes[i:i+batch_size]
                batch_data = self.graph.create_batch(batch_syn)
                logits = self.model(batch_data)
                pred = torch.sigmoid(logits)
                all_preds.append((pred > 0.5).flatten().cpu().numpy())

        all_preds = np.concatenate(all_preds)
        errors = (all_preds != labels.cpu().numpy())
        return errors.mean()

    def save(self, path):
        """Save model weights"""
        torch.save(self.model.state_dict(), path)
        print(f"✓ Model saved to {path}")

    def load(self, path):
        """Load model weights"""
        self.model.load_state_dict(torch.load(path, weights_only=True))
        print(f"✓ Model loaded from {path}")

In [12]:
# ============================================================================
# PART 6: MWPM BASELINE FOR COMPARISON
# ============================================================================

def get_mwpm_accuracy(p, d, num_shots=100000):
    """Compute MWPM accuracy for comparison"""
    import pymatching

    print(f"Computing MWPM baseline ({num_shots:,} shots)...")

    circuit = surface_code_circuit(p, d)
    sampler = circuit.compile_detector_sampler()
    detection_events, observable_flips = sampler.sample(num_shots, separate_observables=True)

    detector_error_model = circuit.detector_error_model(decompose_errors=True)
    matcher = pymatching.Matching.from_detector_error_model(detector_error_model)

    predictions = matcher.decode_batch(detection_events)

    num_errors = sum(
        1 for shot in range(num_shots)
        if not np.array_equal(observable_flips[shot], predictions[shot])
    )

    ler = num_errors / num_shots
    accuracy = 1 - ler
    print(f"✓ MWPM Accuracy: {accuracy:.6f} (LER: {ler:.6f})")

    return accuracy

In [13]:
# ============================================================================
# PART 7: PROGRESSIVE TRAINING - Train until beating MWPM
# ============================================================================

def train_until_beat_mwpm(p, d, device, max_train_size=10**8, chunk_size=10**7):
    """Train with progressively larger datasets until beating MWPM"""
    import gc

    print(f"\n{'#'*60}")
    print(f"# PROGRESSIVE TRAINING: d={d}, p={p}")
    print(f"{'#'*60}")

    # Get MWPM baseline first
    print("\n" + "="*60)
    print("STEP 1: Getting MWPM baseline")
    print("="*60)
    mwpm_accuracy = get_mwpm_accuracy(p, d)
    print(f"\n🎯 Target to beat: {mwpm_accuracy:.6f}")

    # Initialize trainer
    print("\n" + "="*60)
    print("STEP 2: Setting up GNN")
    print("="*60)
    trainer = GNNTrainer(p, d, device)

    # Progressive training
    print("\n" + "="*60)
    print("STEP 3: Progressive training")
    print("="*60)

    train_size = 100
    beat_mwpm = False

    while train_size <= max_train_size and not beat_mwpm:
        print(f"\n{'─'*60}")
        print(f"📊 Training with {train_size:,} samples")
        print(f"{'─'*60}")

        # Reset model for fresh start
        trainer.model = SyndromeDecoder(hidden_dim=128, num_layers=4).to(device)
        trainer.optimizer = torch.optim.Adam(trainer.model.parameters(), lr=3e-4)

        # Calculate chunks needed
        num_chunks = max(1, train_size // chunk_size)
        samples_per_chunk = min(train_size, chunk_size)

        # Train in chunks
        for chunk_idx in range(num_chunks):
            current_chunk_size = min(samples_per_chunk, train_size - chunk_idx * chunk_size)

            print(f"\n  Chunk {chunk_idx + 1}/{num_chunks}: {current_chunk_size:,} samples")
            print(f"  Generating data...")

            syndromes, labels = trainer.sample_data(current_chunk_size)

            # Multiple epochs per chunk for small datasets
            num_epochs = max(1, min(10, 10000 // (current_chunk_size // 256 + 1)))

            for epoch in range(num_epochs):
                loss, acc = trainer.train_epoch(syndromes, labels, batch_size=256)
                print(f"    Epoch {epoch+1}/{num_epochs}: Loss={loss:.4f}, Train Acc={acc:.4f}")

            # Cleanup
            del syndromes, labels
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        # Evaluate
        print(f"\n  Evaluating on fresh test data...")
        gnn_accuracy = trainer.evaluate(num_samples=10000)

        print(f"\n  📈 Results:")
        print(f"     GNN Accuracy:  {gnn_accuracy:.6f}")
        print(f"     MWPM Accuracy: {mwpm_accuracy:.6f}")
        print(f"     Difference:    {(gnn_accuracy - mwpm_accuracy)*100:+.3f}%")

        if gnn_accuracy > mwpm_accuracy:
            print(f"\n  🎉 SUCCESS! GNN beats MWPM at train_size = {train_size:,}")
            beat_mwpm = True
        else:
            print(f"\n  ❌ Not yet. Trying 10x more data...")
            train_size *= 10
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    if not beat_mwpm:
        print(f"\n❌ Did not beat MWPM within max train_size = {max_train_size:,}")
        return None, None

    return trainer, train_size

In [ ]:
# ============================================================================
# PART 8: RUN THE EXPERIMENT
# ============================================================================

import os

# Configuration
train_p = 0.005
max_train_size = 10**8
chunk_size = 10**7

results = {}
trained_models = {}

# Train for each distance
for d in [3, 5, 7]:
    model_path = f"gnn_decoder_d{d}_p{train_p}.pt"

    print(f"\n{'█'*60}")
    print(f"█  DISTANCE {d}")
    print(f"{'█'*60}")

    # Check if model already exists
    if os.path.exists(model_path):
        print(f"\n✓ Found existing model: {model_path}")
        trainer = GNNTrainer(train_p, d, device)
        trainer.load(model_path)
        train_size = None
    else:
        print(f"\n→ No saved model found. Training new model...")
        trainer, train_size = train_until_beat_mwpm(
            p=train_p, d=d, device=device,
            max_train_size=max_train_size, chunk_size=chunk_size
        )

        if trainer is not None:
            trainer.save(model_path)
        else:
            print(f"Training failed for d={d}")
            continue

    trained_models[d] = trainer
    results[d] = train_size

# Print summary
print(f"\n\n{'═'*60}")
print("FINAL RESULTS SUMMARY")
print(f"{'═'*60}")
for d, train_size in results.items():
    if train_size:
        print(f"  Distance {d}: Beat MWPM with {train_size:,} samples")
    else:
        print(f"  Distance {d}: Loaded from saved model")
print(f"{'═'*60}")


████████████████████████████████████████████████████████████
█  DISTANCE 3
████████████████████████████████████████████████████████████

→ No saved model found. Training new model...

############################################################
# PROGRESSIVE TRAINING: d=3, p=0.005
############################################################

STEP 1: Getting MWPM baseline
Computing MWPM baseline (100,000 shots)...
✓ MWPM Accuracy: 0.982730 (LER: 0.017270)

🎯 Target to beat: 0.982730

STEP 2: Setting up GNN

Initializing GNN Trainer for d=3, p=0.005

[1/3] Building quantum circuit...

[2/3] Building graph skeleton...
✓ Graph skeleton built: 24 detectors, 48 connections
  Spatial edges: 32
  Temporal edges: 16
  Average neighbors per detector: 2.0

[3/3] Building neural network...
✓ Model built: 61,697 parameters

Ready to train!


STEP 3: Progressive training

────────────────────────────────────────────────────────────
📊 Training with 100 samples
───────────────────────────────────────

    Epoch 1/10: Loss=1.1775, Train Acc=0.1500


    Epoch 2/10: Loss=1.1569, Train Acc=0.1500


    Epoch 3/10: Loss=1.1419, Train Acc=0.6200


    Epoch 4/10: Loss=1.1297, Train Acc=0.6400


...

    Epoch 9/10: Loss=0.9304, Train Acc=0.6686


    Epoch 10/10: Loss=0.9260, Train Acc=0.6721

  Evaluating on fresh test data...

  📈 Results:
     GNN Accuracy:  0.653100
     MWPM Accuracy: 0.986230
     Difference:    -33.313%

  ❌ Not yet. Trying 10x more data...

────────────────────────────────────────────────────────────
📊 Training with 1,000,000 samples
────────────────────────────────────────────────────────────

  Chunk 1/1: 1,000,000 samples
  Generating data...


    Epoch 1/2: Loss=0.9685, Train Acc=0.6356


    Epoch 2/2: Loss=0.9177, Train Acc=0.6806

  Evaluating on fresh test data...

  📈 Results:
     GNN Accuracy:  0.720600
     MWPM Accuracy: 0.986230
     Difference:    -26.563%

  ❌ Not yet. Trying 10x more data...

────────────────────────────────────────────────────────────
📊 Training with 10,000,000 samples
────────────────────────────────────────────────────────────

  Chunk 1/1: 10,000,000 samples
  Generating data...


Training:  32%|███▏      | 12651/39063 [4:42:15<9:36:10,  1.31s/it, loss=0.8947, acc=0.6692] 

In [ ]:
# ============================================================================
# QUICK TEST - Run this first to verify everything works
# ============================================================================

print("🧪 Quick Test: Training a small GNN on d=3")
print("="*50)

# Create trainer
test_trainer = GNNTrainer(p=0.005, d=3, device=device)

# Generate small dataset
print("\nGenerating 10,000 training samples...")
syndromes, labels = test_trainer.sample_data(10000)
print(f"✓ Syndromes shape: {syndromes.shape}")
print(f"✓ Labels shape: {labels.shape}")
print(f"✓ Example syndrome: {syndromes[0, :5].tolist()} ...")

# Train for 3 epochs
print("\nTraining for 3 epochs...")
for epoch in range(3):
    loss, acc = test_trainer.train_epoch(syndromes, labels, batch_size=256)
    print(f"  Epoch {epoch+1}: Loss={loss:.4f}, Accuracy={acc:.4f}")

# Evaluate
print("\nEvaluating on fresh test data...")
test_acc = test_trainer.evaluate(5000)
print(f"✓ Test Accuracy: {test_acc:.4f}")

print("\n" + "="*50)
print("✅ Quick test passed! The GNN is working.")
print("="*50)

🧪 Quick Test: Training a small GNN on d=3

Initializing GNN Trainer for d=3, p=0.005

[1/3] Building quantum circuit...

[2/3] Building graph skeleton...
✓ Graph skeleton built: 24 detectors, 48 connections
  Average neighbors per detector: 2.0

[3/3] Building neural network...
✓ Model built: 59,137 parameters

Ready to train!


Generating 10,000 training samples...
✓ Syndromes shape: torch.Size([10000, 24])
✓ Labels shape: torch.Size([10000])
✓ Example syndrome: [-1, -1, -1, -1, -1] ...

Training for 3 epochs...


  Epoch 1: Loss=0.4129, Accuracy=0.8907


  Epoch 2: Loss=0.2895, Accuracy=0.8960


  Epoch 3: Loss=0.2861, Accuracy=0.8960

Evaluating on fresh test data...
✓ Test Accuracy: 0.9024

✅ Quick test passed! The GNN is working.
